# Libraries Import and Base Path initialization

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import boxmot.trackers.ocsort.ocsort as o
print(dir(o))

from boxmot import Boxmot
help(Boxmot)

from boxmot.trackers.ocsort.ocsort import OcSort
help(OcSort)

## Osnet location

Downloading...
From: https://drive.google.com/uc?id=1sSwXSUlj4_tHZequ_iZ8w_Jh0VaRQMqF

To: /Users/julianyang/Mine/Kuliah/SMT 6/CV/UAS/.venv/lib/python3.10/site-packages/models/osnet_x0_25_msmt17.pt

100%|██████████| 3.06M/3.06M [00:00<00:00, 15.4MB/s]
SUCCESS  | Loaded pretrained weights from /Users/julianyang/Mine/Kuliah/SMT 6/CV/UAS/.venv/lib/python3.10/site-packages/models/osnet_x0_25_msmt17.pt

In [1]:
import os, cv2, json, math, pickle, random
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
from ultralytics import YOLO
from boxmot.trackers.ocsort.ocsort import OcSort
# from boxmot.trackers.deepocsort.deepocsort import DeepOcSort
import torch

from mmengine.config import Config
from mmengine.registry import MODELS
from mmengine.runner import load_checkpoint
from mmaction.apis import init_recognizer

# # This function will now work correctly because we are running from the cloned directory
from mmaction.utils import register_all_modules
register_all_modules(init_default_scope=True) # We set the scope manually later

#Colab Base Path
# base_path = "/content/drive/MyDrive/SMT 6/CV/UAS"

#Local Base Path
base_path = ""

# === Dataset Paths ===
# data_path = os.path.join(base_path, "match_videos")
data_path = os.path.join(base_path, "practice_videos")

# video_path = os.path.join(data_path, "Knee(Bryan) vs Double(Law) TWT 2024.mp4")
# video_path = os.path.join(data_path, "Knee(Bryan) vs Double(Law) 1 round.mp4")
# video_path = os.path.join(data_path, "lowhigh(bryan) vs ninjakilla(law)_1round.mp4")
video_path = os.path.join(data_path, "Bryan_LR_Complete.mp4")

# annotation_path = os.path.join(base_path, "match_videos/Knee(Bryan) vs Double(Law) TWT 2024.json")
# annotation_path = os.path.join(data_path, "Knee_reindexed.json")
annotation_path = os.path.join(data_path, "Bryan_LR_Complete.json")

labels_path = os.path.join(data_path, "move_labels.json") # move class labels
skeleton_dataset = os.path.join(data_path, "skeleton_dataset.pkl") # STGCN++ dataset
output_dir = os.path.join(data_path, "frames")
kp_dir = os.path.join(data_path)

# video_path = os.path.join(base_path, "Bryan_2/Bryan_15_move_trimmed.mp4")
# annotation_path = os.path.join(base_path, "Bryan_2/Bryan_15_move_2.json")
# output_dir = os.path.join(base_path, "Bryan_2/frames")

# === Load YOLO detection and pose models ===
# yolo = YOLO("yolo11s.pt")
# yolo = YOLO("yolo11m.pt")
# yolo = YOLO("yolo26s.pt")
yolo  = YOLO("runs/detect/practice_m/weights/best.pt")
# yolo  = YOLO("twt_practice_m.pt")
# yolo_pose = YOLO("yolo11n-pose.pt")
yolo_pose = YOLO("yolo11s-pose.pt")
yolo.to("mps")
yolo_pose.to("mps")

# os.makedirs(output_dir, exist_ok=True)

/Users/julianyang/Mine/Kuliah/SMT 6/CV/UAS/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


YOLO(
  (model): PoseModel(
    (model): Sequential(
      (0): Conv(
        (conv): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (1): Conv(
        (conv): Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (2): C3k2(
        (cv1): Conv(
          (conv): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
          (act): SiLU(inplace=True)
        )
        (cv2): Conv(
          (conv): Conv2d(96, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(128, eps=0.001, momentum=0.03, affine=True, track_running_sta

# YOLO and ByteTrack character tracking

In [2]:
def make_tracker():
    return OcSort(
        half=True,
        device="mps",
        max_age=90,
        min_hits=2,
        iou_threshold=0.15,
        det_thresh=0.20,
        # iou_threshold=0.25,
        # det_thresh=0.30,
    )

tracker = make_tracker()

def yolo_detect(image):
    result = yolo(image, conf=0.5, iou=0.35, classes=[0])[0]

    # the results of yolo.predict contains list of object per frame it detects, for example image will have 1 object in the list
    # while video will have as many object in it as the video frames
    # we will access the first object as it is an image
    # Play with conf(minimum conf to be detected) and 
    # iou (how much the boxes can overlap to be considered the same object)
    if result.boxes is None or len(result.boxes) == 0:
        return np.empty((0, 6), dtype=np.float32)

    xyxy = result.boxes.xyxy.cpu().numpy() # convert boxes x1, y1, x2, y2 of selected object to numpy, then to int
    conf = result.boxes.conf.cpu().numpy().reshape(-1, 1) # convert boxes confidence of selected object to numpy
    cls = result.boxes.cls.cpu().numpy().reshape(-1, 1) # convert boxes class of selected object to numpy

    detections = np.hstack((xyxy, conf, cls))
    # stack boxes, conf, cls horizontally (it only accepts tuple so we encapsulate it with double ()

    # print("Results xyxy", result.boxes.xyxy) # print the x1, y1, x2, y2 from boxes of the object
    # print("detections", detections, type(detections))
    return detections

def ocsort_tracking(detections, image): 
    if detections.shape[0] == 0:
        return None
    
    tracks = tracker.update(detections, image)
    if tracks is None or len(tracks) == 0:
        return None
    
    ids = tracks[:, 4].astype(int).reshape(-1, 1)
    boxes = tracks[:, :4].astype(int)
    id_box_array = np.hstack((ids, boxes))
    return id_box_array

def pad_box(h, w, box, padding=25):
    # box: [id, x1, y1, x2, y2]
    _, x1, y1, x2, y2 = box
    x1 = max(0, x1 - padding)
    y1 = max(0, y1 - padding)
    x2 = min(w, x2 + padding)
    y2 = min(h, y2 + padding)

    return np.array([_, x1, y1, x2, y2], dtype=int)
    
def crop_roi(frame, box):
    # box: [id, x1, y1, x2, y2]
    _, x1, y1, x2, y2 = box
    # print("x1: ", x1, "y1: ", y1, "x2: ", x2, "y2: ", y2)
    return frame[y1:y2, x1:x2]

    # Matrix (Memory) Coordinates: NumPy thinks in (row, column).
    # Because images are processed top-to-bottom, a row corresponds to 
    # the vertical y position, and a column corresponds to the horizontal x position.

def display_roi(player1_roi, player2_roi):
    fig, axes = plt.subplots(nrows = 1, ncols = 2, figsize = (4, 8))
    axes[0].imshow(player1_roi)
    axes[1].imshow(player2_roi)

    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

def is_timer_missing(frame, edge_threshold=50):
    """
    Checks the Tekken timer UI using Edge Detection.
    Returns True if the sharp metallic borders of the numbers are missing.
    """
    y1, y2 = 26, 86
    x1, x2 = 600, 680
    timer_roi = frame[y1:y2, x1:x2]
    
    gray = cv2.cvtColor(timer_roi, cv2.COLOR_BGR2GRAY)
    
    # cv2.Canny highlights sharp transitions. 
    # The silver border of the font will light up brilliantly here.
    edges = cv2.Canny(gray, 100, 200)
    
    # Count how many 'edge' pixels exist in that small box
    edge_count = cv2.countNonZero(edges)
    
    # If the count drops below the threshold, the timer is gone.
    return edge_count < edge_threshold

def is_cinematic_zoom(boxes, frame_height, threshold=0.75): # Raised to 85%
    """
    Detects if the camera has zoomed in brutally for a Rage Art, Tornado, or K.O.
    """
    if boxes is None or len(boxes) == 0:
        return False
        
    for box in boxes:
        box_h = box[3] - box[1] 
        if box_h > (frame_height * threshold):
            return True
            
    return False

WARNING  Max age > max observations, increasing size of max observations...

INFO     OcSort: det_thresh=0.2, max_age=90, max_obs=50, min_hits=2, iou_threshold=0.15, per_class=False,          
         asso_func=iou, min_conf=0.1, delta_t=3, inertia=0.2, use_byte=False, Q_xy_scaling=0.01, Q_s_scaling=0.0001

# YOLO Pose Function

In [3]:
def empty_keypoints():
    return np.full((17, 3), np.nan, dtype=np.float32)

def run_yolopose(image, input_size=192, conf=0.25, iou=0.35):
    if image is None or image.size == 0:
        return empty_keypoints()

    roi_height, roi_width = image.shape[:2]
    if roi_height == 0 or roi_width == 0:
        return empty_keypoints()

    result = yolo_pose(image, imgsz=input_size, conf=conf, iou=iou, verbose=False)[0]
    if result.keypoints is None or len(result.keypoints) == 0:
        return empty_keypoints()

    xy = result.keypoints.xy.cpu().numpy()
    kp_conf = result.keypoints.conf
    if kp_conf is None:
        scores = np.ones(xy.shape[:2], dtype=np.float32)
    else:
        scores = kp_conf.cpu().numpy()

    if xy.shape[0] == 0:
        return empty_keypoints()

    if result.boxes is not None and len(result.boxes) > 0:
        best_idx = int(np.argmax(result.boxes.conf.cpu().numpy()))
        best_idx = min(best_idx, xy.shape[0] - 1)
    else:
        mean_scores = np.nanmean(scores, axis=1)
        best_idx = 0 if np.all(np.isnan(mean_scores)) else int(np.nanargmax(mean_scores))

    keypoints = empty_keypoints()
    points_xy = xy[best_idx]
    point_scores = scores[best_idx]
    num_points = min(17, points_xy.shape[0])

    keypoints[:num_points, 0] = points_xy[:num_points, 1] / roi_height
    keypoints[:num_points, 1] = points_xy[:num_points, 0] / roi_width
    keypoints[:num_points, 2] = point_scores[:num_points]
    return keypoints

def denormalize_points(points, original_height, original_width, input_size=None):
    """
    Converts normalized YOLO-pose ROI keypoints back to ROI pixel coordinates.
    """
    y, x, c = points
    if np.isnan(y) or np.isnan(x):
        return (np.nan, np.nan, c)

    y_abs = y * original_height
    x_abs = x * original_width
    return (y_abs, x_abs, c)

def normalize_points_to_full_frame(kp_array, box, full_height, full_width):
    """
    Normalize array of keypoints to full frame size
    """
    _, x1, y1, x2, y2 = box
    roi_height = y2-y1
    roi_width = x2-x1

    kp_full = []
    for kp in kp_array:
        y_roi, x_roi, c = denormalize_points(kp, roi_height, roi_width)
        if np.isnan(y_roi) or np.isnan(x_roi):
            kp_full.append([np.nan, np.nan, c])
            continue

        y_full = (y_roi + y1) / full_height
        x_full = (x_roi + x1) / full_width
        kp_full.append([y_full, x_full, c])

    return np.array(kp_full)
        

def draw_keypoints(frame, keypoints, color, full_height, full_width, score_threshold=0.0):
    if keypoints is None:
        return

    for kp in keypoints:
        y, x, c = kp
        if not np.isfinite(y) or not np.isfinite(x):
            continue
        if np.isfinite(c) and c < score_threshold:
            continue

        y_px = int(np.clip(y * full_height, 0, full_height - 1))
        x_px = int(np.clip(x * full_width, 0, full_width - 1))
        cv2.circle(frame, (x_px, y_px), 3, color, thickness=2, lineType=cv2.LINE_AA)


def interpolate_points(player_kp):
    player_kp = np.array(player_kp)
    for kp in range(player_kp.shape[1]):
        for coord in range(player_kp.shape[2]):
            data = player_kp[:, kp, coord]
            nans = np.isnan(data) # nans mask example: [true, false, true] based on the positions
            if np.any(~nans):
                data[nans] = np.interp(np.flatnonzero(nans), np.flatnonzero(~nans), data[~nans])
            player_kp[:, kp, coord] = data
            print(data)
    return player_kp
# for kp in player1_kp:
#     print(kp)
#     y, x, c = denormalize_points(kp, original_height, original_width)

# YOLO Pose Extraction

In [4]:
input_size = 192
cap = cv2.VideoCapture(video_path)
print(video_path, os.path.exists(video_path))

player1_kp = []
player2_kp = []
player1_track = []  # [id, x1, y1, x2, y2] or NaNs per frame
player2_track = []  # [id, x1, y1, x2, y2] or NaNs per frame
other_track = []    # [id, x1, y1, x2, y2] or NaNs per frame
player_set = False
player1_id, player2_id = None, None
player1_box, player2_box, other_box = None, None, None
player1_kp_full, player2_kp_full, other_kp_full = None, None, None
frame_count = 0

missing_timer_frames = 0
buffer_limit = 10  # 10 frames ignores brief juggles, but catches the 15-frame practice reset
tracking_active = True

cinematic_frames = 0
cinematic_buffer = 5 # Wait 5 frames to confirm a zoom

while cap.isOpened(): # read every single frame of the video
    ret, frame_bgr = cap.read()
    if not ret:
        print("End of video")
        break

    original_height, original_width = frame_bgr.shape[:2]

    # --- 1. CHECK THE TIMER ---
    if is_timer_missing(frame_bgr):
        missing_timer_frames += 1
    else:
        missing_timer_frames = 0
        tracking_active = True # Timer is clearly visible, tracking is safe

    # --- HANDLE THE RESET STATE ---
    if missing_timer_frames > buffer_limit:
        
        # Only print and wipe memory the FIRST time we cross the threshold
        if tracking_active:
            print(f"Scene transition detected at frame {frame_count}. Pausing tracking...")
            tracking_active = False 
            
            tracker = make_tracker()
            # last_p1_box = None
            # last_p2_box = None
            # If using an OC-SORT instance, destroy/re-init it here.
            player_set = False

        # We are in a reset state (black screen or waiting for fade-in).
        # Append empty frames to keep your temporal arrays perfectly aligned for the pipeline.
        player1_kp.append(np.full((17, 3), np.nan))
        player2_kp.append(np.full((17, 3), np.nan))
        player1_track.append(np.full(5, np.nan, dtype=np.float32))
        player2_track.append(np.full(5, np.nan, dtype=np.float32))
        other_track.append(np.full(5, np.nan, dtype=np.float32))
        
        frame_count += 1
        
        # Skip the rest of the loop entirely. Do not run YOLO.
        continue


    # --- 2. THE CINEMATIC CHECK ---
    # Run your raw YOLO detection ONCE per frame
    detections = yolo_detect(frame_bgr) 
    
    # # Check if the boxes are massive (camera zoomed in)
    # if is_cinematic_zoom(detections, frame_height, threshold=0.85):
    #     cinematic_frames += 1
    # else:
    #     cinematic_frames = 0

    # if cinematic_frames > cinematic_buffer:
    #     print(f"Cinematic zoom detected at frame {frame_count}. Pausing tracking...")
        
    #     # Append NaNs because no actual gameplay is happening
    #     player1_kp.append(np.full((17, 3), np.nan))
    #     player2_kp.append(np.full((17, 3), np.nan))
        
    #     # Wipe the player memory! 
    #     # Characters often land in different spots after a Tornado/Rage Art.
    #     # This forces the logic to re-evaluate who is on the left/right when zooming out.
    #     player_set = False 
        
    #     frame_count += 1
    #     continue # Skip OC-SORT and YOLO pose

    # frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    id_box_array = ocsort_tracking(detections, frame_bgr) # 2d array containing id_box from p1 and 2
    # print(id_box_array)

    if id_box_array is None or id_box_array.size == 0:
        player1_kp.append(np.full((17, 3), np.nan))
        player2_kp.append(np.full((17, 3), np.nan))
        player1_track.append(np.full(5, np.nan, dtype=np.float32))
        player2_track.append(np.full(5, np.nan, dtype=np.float32))
        other_track.append(np.full(5, np.nan, dtype=np.float32))
        frame_count += 1
        continue

    # if frame_count == 0:
    #     original_height, original_width = frame_bgr.shape[:2]
    #     print("Original height, original_width", original_height, original_width)
    #     player1_id = id_box_array[0, 0]
    #     player2_id = id_box_array[1, 0]

    if id_box_array is not None and id_box_array.shape[0] >= 2 and not player_set:
        centers_x = (id_box_array[:,1] + id_box_array[:,3]) / 2
        order = np.argsort(centers_x)           # left -> right
        player1_id = int(id_box_array[order[0], 0])
        player2_id = int(id_box_array[order[1], 0]) 

        # ABOVE EXTRA STEPS MIGHT NOT BE NECESSARY
        # player1_id = id_box_array[0, 0]
        # player2_id = id_box_array[1, 0]
        player_set = True

    p1_exist = any(id_box_array[:, 0] == player1_id)
    p2_exist = any(id_box_array[:, 0] == player2_id)

    other_mask = ~np.isin(id_box_array[:, 0], [player1_id, player2_id])
    has_other = np.any(other_mask)
    player1_track_frame = np.full(5, np.nan, dtype=np.float32)
    player2_track_frame = np.full(5, np.nan, dtype=np.float32)
    other_track_frame = np.full(5, np.nan, dtype=np.float32)
    
    if p1_exist:
        player1_box = id_box_array[id_box_array[:, 0] == player1_id][0]
        player1_track_frame = player1_box.astype(np.float32)
        # print("p1 box", player1_box)
        player1_box_padded = pad_box(original_height, original_width, player1_box)
        player1_roi = crop_roi(frame_bgr, player1_box_padded)
        player1_kp_raw = run_yolopose(player1_roi, input_size)
        player1_kp_full = normalize_points_to_full_frame(player1_kp_raw, player1_box_padded, original_height, original_width)
        player1_kp.append(player1_kp_full)
    else:
        player1_box = None
        player1_box_padded = None
        player1_kp_full = None
        player1_kp.append(np.full((17, 3), np.nan))

    if p2_exist:
        player2_box = id_box_array[id_box_array[:, 0] == player2_id][0]
        player2_track_frame = player2_box.astype(np.float32)
        player2_box_padded = pad_box(original_height, original_width, player2_box)
        player2_roi = crop_roi(frame_bgr, player2_box_padded)
        player2_kp_raw = run_yolopose(player2_roi, input_size)
        player2_kp_full = normalize_points_to_full_frame(player2_kp_raw, player2_box_padded, original_height, original_width)
        player2_kp.append(player2_kp_full)
    else:
        player2_box = None
        player2_box_padded = None
        player2_kp_full = None
        player2_kp.append(np.full((17, 3), np.nan))

    if has_other:
        other_box = id_box_array[other_mask][0]
        other_track_frame = other_box.astype(np.float32)
        other_box_padded = pad_box(original_height, original_width, other_box)
        other_roi = crop_roi(frame_bgr, other_box_padded)
        other_kp_raw = run_yolopose(other_roi, input_size)
        other_kp_full = normalize_points_to_full_frame(other_kp_raw, other_box_padded, original_height, original_width)
    else:
        other_box = None
        other_box_padded = None
        other_kp_full = None


    # Visualization: draw player 1 (green) and player 2 (red)
    color_p1 = (0, 255, 0)  # green (B, G, R)
    color_p2 = (0, 0, 255)  # red
    
    # Boxes + IDs
    if player1_box is not None:
        obj_id, x1, y1, x2, y2 = player1_box_padded
        cv2.rectangle(frame_bgr, (x1, y1), (x2, y2), color_p1, thickness=2, lineType=cv2.LINE_AA)
        cv2.putText(frame_bgr, "ID: " + str(int(obj_id)), (x1, y1+20), cv2.FONT_HERSHEY_SIMPLEX,
                    fontScale=1, color=color_p1, thickness=1, lineType=cv2.LINE_AA)
    
    if player2_box is not None:
        obj_id, x1, y1, x2, y2 = player2_box_padded
        cv2.rectangle(frame_bgr, (x1, y1), (x2, y2), color_p2, thickness=2, lineType=cv2.LINE_AA)
        cv2.putText(frame_bgr, "ID: " + str(int(obj_id)), (x1, y1+20), cv2.FONT_HERSHEY_SIMPLEX,
                    fontScale=1, color=color_p2, thickness=1, lineType=cv2.LINE_AA)
    
    if other_box is not None:
        obj_id, x1, y1, x2, y2 = other_box_padded
        cv2.rectangle(frame_bgr, (x1, y1), (x2, y2), (255, 0, 0), thickness=2, lineType=cv2.LINE_AA)
        cv2.putText(frame_bgr, "ID: " + str(int(obj_id)), (x1, y1+20), cv2.FONT_HERSHEY_SIMPLEX,
                    fontScale=1, color=color_p2, thickness=1, lineType=cv2.LINE_AA)
    
    # Keypoints
    draw_keypoints(frame_bgr, player1_kp_full, color_p1, original_height, original_width)
    draw_keypoints(frame_bgr, player2_kp_full, color_p2, original_height, original_width)

    player1_track.append(player1_track_frame)
    player2_track.append(player2_track_frame)
    other_track.append(other_track_frame)

    cv2.imshow('Process feed', frame_bgr)
    if cv2.waitKey(25) & 0xFF == ord('q'):
        break

    # frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    # plt.imshow(frame_rgb)
    # plt.show()

    frame_count += 1

cap.release()

player1_kp = interpolate_points(player1_kp)
player2_kp = interpolate_points(player2_kp)
player1_track = np.asarray(player1_track, dtype=np.float32)
player2_track = np.asarray(player2_track, dtype=np.float32)
other_track = np.asarray(other_track, dtype=np.float32)

np.save(os.path.join(kp_dir, "player1_kp"), player1_kp)
np.save(os.path.join(kp_dir, "player2_kp"), player2_kp)
np.save(os.path.join(kp_dir, "player1_track"), player1_track)
np.save(os.path.join(kp_dir, "player2_track"), player2_track)
np.save(os.path.join(kp_dir, "other_track"), other_track)

practice_videos/Bryan_LR_Complete.mp4 True

0: 384x640 2 fighters, 210.1ms
Speed: 7.3ms preprocess, 210.1ms inference, 47.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 fighters, 29.0ms
Speed: 2.1ms preprocess, 29.0ms inference, 7.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 fighters, 20.0ms
Speed: 1.7ms preprocess, 20.0ms inference, 7.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 fighters, 18.8ms
Speed: 1.8ms preprocess, 18.8ms inference, 6.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 fighters, 19.1ms
Speed: 1.6ms preprocess, 19.1ms inference, 5.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 fighters, 20.1ms
Speed: 1.8ms preprocess, 20.1ms inference, 7.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 fighters, 20.6ms
Speed: 1.7ms preprocess, 20.6ms inference, 6.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 fighters, 21.1ms
Speed: 1.8ms preprocess, 21

WARNING  Max age > max observations, increasing size of max observations...

INFO     OcSort: det_thresh=0.2, max_age=90, max_obs=50, min_hits=2, iou_threshold=0.15, per_class=False,          
         asso_func=iou, min_conf=0.1, delta_t=3, inertia=0.2, use_byte=False, Q_xy_scaling=0.01, Q_s_scaling=0.0001


0: 384x640 2 fighters, 24.8ms
Speed: 1.5ms preprocess, 24.8ms inference, 8.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 fighters, 22.4ms
Speed: 2.8ms preprocess, 22.4ms inference, 6.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 fighters, 21.7ms
Speed: 1.6ms preprocess, 21.7ms inference, 7.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 fighters, 21.1ms
Speed: 2.1ms preprocess, 21.1ms inference, 6.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 fighters, 20.9ms
Speed: 1.9ms preprocess, 20.9ms inference, 6.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 fighters, 21.2ms
Speed: 1.7ms preprocess, 21.2ms inference, 7.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 fighters, 21.1ms
Speed: 1.8ms preprocess, 21.1ms inference, 7.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 fighters, 22.1ms
Speed: 1.7ms preprocess, 22.1ms inference, 7.5ms postprocess per image at

WARNING  Max age > max observations, increasing size of max observations...

...

INFO     OcSort: det_thresh=0.2, max_age=90, max_obs=50, min_hits=2, iou_threshold=0.15, per_class=False,          
         asso_func=iou, min_conf=0.1, delta_t=3, inertia=0.2, use_byte=False, Q_xy_scaling=0.01, Q_s_scaling=0.0001


0: 384x640 2 fighters, 23.6ms
Speed: 1.8ms preprocess, 23.6ms inference, 9.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 fighters, 20.4ms
Speed: 1.5ms preprocess, 20.4ms inference, 6.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 fighters, 18.4ms
Speed: 1.7ms preprocess, 18.4ms inference, 6.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 fighters, 18.5ms
Speed: 1.6ms preprocess, 18.5ms inference, 6.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 fighters, 18.6ms
Speed: 1.8ms preprocess, 18.6ms inference, 6.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 fighters, 18.7ms
Speed: 1.5ms preprocess, 18.7ms inference, 6.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 fighters, 18.5ms
Speed: 1.7ms preprocess, 18.5ms inference, 5.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 fighters, 19.3ms
Speed: 1.5ms preprocess, 19.3ms inference, 6.2ms postprocess per image at

WARNING  Max age > max observations, increasing size of max observations...

INFO     OcSort: det_thresh=0.2, max_age=90, max_obs=50, min_hits=2, iou_threshold=0.15, per_class=False,          
         asso_func=iou, min_conf=0.1, delta_t=3, inertia=0.2, use_byte=False, Q_xy_scaling=0.01, Q_s_scaling=0.0001


0: 384x640 2 fighters, 22.4ms
Speed: 1.7ms preprocess, 22.4ms inference, 7.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 fighters, 19.4ms
Speed: 1.3ms preprocess, 19.4ms inference, 7.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 fighters, 19.9ms
Speed: 1.6ms preprocess, 19.9ms inference, 6.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 fighters, 20.1ms
Speed: 1.8ms preprocess, 20.1ms inference, 6.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 fighters, 20.2ms
Speed: 1.3ms preprocess, 20.2ms inference, 6.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 fighters, 19.3ms
Speed: 1.3ms preprocess, 19.3ms inference, 7.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 fighters, 19.5ms
Speed: 2.1ms preprocess, 19.5ms inference, 6.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 fighters, 20.1ms
Speed: 1.9ms preprocess, 20.1ms inference, 6.8ms postprocess per image at

Sure! Here's a concise summary of everything we discussed, formatted in markdown for easy reference:

---

## 📚 Summary: MMAction2 Skeleton Dataset Format & Preparation Steps

Skeleton-based Action Recognition in MMAction2 doesn’t require splitting the original video, but it **does require splitting the keypoint data** into action-based segments.

---

### 🧬 Dataset Format Overview (`.pkl`)

```python
{
  "split": {
    "train": ["clip1", "clip2", ...],
    "val": ["clip7", "clip8", ...],
    ...
  },
  "annotations": [
    {
      "frame_dir": "clip1",
      "label": 0,
      "img_shape": (1080, 1920),
      "original_shape": (1080, 1920),
      "total_frames": 87,
      "keypoint": np.ndarray([M, T, V, C]),
      "keypoint_score": np.ndarray([M, T, V])
    },
    ...
  ]
}
```

- **`frame_dir`**: Unique name for each clip
- **`label`**: Action class (int)
- **`img_shape` & `original_shape`**: Optional frame resolution
- **`total_frames`**: Frames in the segment
- **`keypoint`**: Shape `[M x T x V x C]` (people, frames, joints, coords)
- **`keypoint_score`**: Confidence for each keypoint `[M x T x V]`

---

### ⚙️ Steps to Prepare from a Long Video

If you already extracted full video keypoints:

1. **Use Annotations**  
   Get frame ranges for each action from your annotation file.

2. **Slice Keypoint Arrays**  
   Extract each action clip from the full keypoint array using its frame indices.

3. **Assign Clip Identifiers**  
   Name each segment like `clip001`, `clip002`, etc.

4. **Group into Splits**  
   Organize clip names into `'train'`, `'val'`, etc. inside the `split` dictionary.

5. **Build Annotations List**  
   For each clip, create a dictionary with all required fields and add it to `annotations`.

6. **Save to Pickle**  
   Combine `split` and `annotations` into a Python dict and save as `.pkl`.

---

Want me to build a sample Python script to help automate these steps? Happy to dive in! 💻

# Prepare dataloader

In [5]:
# JSON Annotation
with open(annotation_path) as f:
    annotations = json.load(f)

with open(labels_path) as l:
    labels = json.load(l)

# Keypoints
player1_kp = np.load(os.path.join(kp_dir, "player1_kp.npy"))
player2_kp = np.load(os.path.join(kp_dir, "player2_kp.npy"))

# STGCN++ compliant dataset structure
datasets = {
    "split": {
        "train": [],
        "val": []
    },
    "annotations": []
}

# sequences_id = list(annotations.keys())[0]
# data = annotations[sequences_id]

# print(sequences_id, data)
# print(annotations.keys())
# print(player1_kp[0])

def kp_slicing(kp):
    conf = kp[:, :, 2].astype(np.float32)
    x_coord = kp[:, :, 1]
    y_coord = kp[:, :, 0]
    coords = np.stack((x_coord, y_coord), axis = -1).astype(np.float32)
    # print("confidence:", conf.shape)
    # print("xy:", coords.shape)

    # Expand dimensions to meet MMAction2 requirements
    # keypoint expected: [M, T, V, C] -> (1, Frames, 17, 2)
    keypoint_mm = np.expand_dims(coords, axis=0)
    
    # keypoint_score expected: [M, T, V] -> (1, Frames, 17)
    score_mm = np.expand_dims(conf, axis=0).astype(np.float32)
    
    return keypoint_mm, score_mm

def train_val_split(annotations):
    random.shuffle(annotations)
    split_idx = int(1 * len(annotations))
    train_dirs = [ann["frame_dir"] for ann in annotations[:split_idx]]
    val_dirs = [ann["frame_dir"] for ann in annotations[split_idx:]]
    return train_dirs, val_dirs

for sequence_id, data in annotations.items():
    # print(sequence_id)
    # print(data)
    player = data["player"]
    start_frame = data["start_frame"]
    end_frame = data["end_frame"] + 1  # +1 IS TEMPT FIX FOR TWT ANOTATION

    # print(start_frame, end_frame)

    # The desired data is [person [frames of the actions [kp of frame [coord of kp]]]]
    if player == "player1":
        kp_sequence = player1_kp[start_frame:end_frame]
        # print(kp_sequence.shape)
        kp, kp_score = kp_slicing(kp_sequence)
        # print(kp.dtype, kp_score.dtype)
        move_class = data['character'] + " " + data['move']
        label = labels[move_class]
        # print("label:", label)
        clip_annotation = {
            "frame_dir": sequence_id,
            "label": label,
            "img_shape": (720, 1280),
            "original_shape": (720, 1280),
            "total_frames": len(kp_sequence),
            "keypoint": kp,
            "keypoint_score": kp_score
        }
        datasets["annotations"].append(clip_annotation)
                                       
    else:
        kp_sequence = player2_kp[start_frame:end_frame]
        # print(kp_sequence.shape)
        kp, kp_score = kp_slicing(kp_sequence)
        move_class = data['character'] + " " + data['move']
        label = labels[move_class]
        # print("label:", label)
        clip_annotation = {
            "frame_dir": sequence_id,
            "label": label,
            "img_shape": (720, 1280),
            "original_shape": (720, 1280),
            "total_frames": len(kp_sequence),
            "keypoint": kp,
            "keypoint_score": kp_score
        }
        datasets["annotations"].append(clip_annotation)

train, val = train_val_split(datasets["annotations"])

datasets["split"]['train'].extend(train)
datasets["split"]['train'].extend(val)

with open(skeleton_dataset, "wb") as f:
    pickle.dump(datasets, f)

# Prepare STGCN++ model

In [6]:
config_file = "https://github.com/open-mmlab/mmaction2/blob/main/configs/skeleton/stgcnpp/stgcnpp_8xb16-joint-motion-u100-80e_ntu60-xsub-keypoint-2d.py"
checkpoint = "https://download.openmmlab.com/mmaction/v1.0/skeleton/stgcnpp/stgcnpp_8xb16-joint-motion-u100-80e_ntu60-xsub-keypoint-2d/stgcnpp_8xb16-joint-motion-u100-80e_ntu60-xsub-keypoint-2d_20221228-19a34aba.pth"

# Loading data back
with open(skeleton_dataset, 'rb') as f:
    restored_data = pickle.load(f)

print(restored_data)


{'split': {'train': ['p2_sequence_149', 'p2_sequence_50', 'p1_sequence_42', 'p2_sequence_152', 'p1_sequence_71', 'p2_sequence_27', 'p1_sequence_48', 'p1_sequence_79', 'p2_sequence_126', 'p1_sequence_44', 'p2_sequence_151', 'p1_sequence_5', 'p1_sequence_76', 'p2_sequence_124', 'p1_sequence_36', 'p2_sequence_87', 'p2_sequence_156', 'p1_sequence_74', 'p1_sequence_15', 'p1_sequence_7', 'p1_sequence_131', 'p1_sequence_103', 'p2_sequence_63', 'p2_sequence_58', 'p2_sequence_56', 'p2_sequence_83', 'p2_sequence_19', 'p2_sequence_160', 'p1_sequence_12', 'p2_sequence_82', 'p2_sequence_118', 'p2_sequence_146', 'p2_sequence_128', 'p1_sequence_107', 'p1_sequence_138', 'p1_sequence_139', 'p1_sequence_40', 'p1_sequence_16', 'p1_sequence_75', 'p2_sequence_127', 'p2_sequence_125', 'p1_sequence_144', 'p1_sequence_39', 'p2_sequence_20', 'p1_sequence_3', 'p2_sequence_115', 'p2_sequence_94', 'p1_sequence_45', 'p1_sequence_73', 'p1_sequence_129', 'p2_sequence_64', 'p1_sequence_105', 'p2_sequence_84', 'p2_seq

# Train STGCN++

In [7]:
!cd "/Users/julianyang/Mine/Kuliah/SMT 6/CV/UAS/mmaction2"

!PYTORCH_ENABLE_MPS_FALLBACK=1 CUDA_VISIBLE_DEVICES=-1 ../.venv/bin/python tools/train.py \
  configs/skeleton/stgcnpp/tekken_stgcn.py \
  --work-dir work_dirs/tekken_stgcn_v2 \
  --no-validate

zsh:1: no such file or directory: ../.venv/bin/python
